# Stages

## Overview

This page covers repeating part of a computation over many values, such as a list of files.

A workflow often has a part that runs once, such as loading a calibration or building a mask, and a part that runs many times: per file, per channel of a file, or per chunk of a stream.
The results of the repeated part are then combined, and the rest of the workflow runs once on the combined value.

Sciline expresses this with [Stage](../generated/classes/sciline.Stage.rst), which cuts a flat pipeline into the part that runs once and the part that runs many times.
Accumulators combine the results of the repeated part.
Nothing is added to the pipeline itself.

## A stage

We start with a pipeline similar to the one in [Getting Started](getting-started.ipynb).
Loading the calibration stands in for an expensive step that does not depend on the file.
The provider appends to a list so we can count how often it runs:

In [ ]:
import math
import operator
from typing import NewType

import sciline

_fake_filesystem = {
    'file102.txt': [1.0, 2.0, float('nan'), 3.0],
    'file103.txt': [1.0, 2.0, 3.0, 4.0],
    'file104.txt': [1.0, 2.0, 3.0, 4.0, 5.0],
    'file105.txt': [1.0, 2.0, 3.0],
    'calib_2025.txt': [0.5],
    'calib_2026.txt': [0.25],
}

# 1. Define domain types

Filename = NewType('Filename', str)
RawData = NewType('RawData', list)
CalibrationFilename = NewType('CalibrationFilename', str)
Calibration = NewType('Calibration', float)
Calibrated = NewType('Calibrated', list)
Total = NewType('Total', float)
ScaleFactor = NewType('ScaleFactor', float)
Result = NewType('Result', float)

# 2. Define providers

calibration_loads = []


def load(filename: Filename) -> RawData:
    """Load the data from a file."""
    return RawData(_fake_filesystem[filename])


def load_calibration(filename: CalibrationFilename) -> Calibration:
    """Load the calibration factor."""
    calibration_loads.append(filename)
    return Calibration(_fake_filesystem[filename][0])


def calibrate(raw: RawData, calibration: Calibration) -> Calibrated:
    """Drop NaNs and apply the calibration."""
    return Calibrated([x * calibration for x in raw if not math.isnan(x)])


def total(data: Calibrated) -> Total:
    """Sum the calibrated data."""
    return Total(sum(data))


def scale(total: Total, factor: ScaleFactor) -> Result:
    """Scale the total."""
    return Result(total * factor)


# 3. Create pipeline

providers = [load, load_calibration, calibrate, total, scale]
params = {CalibrationFilename: 'calib_2025.txt', ScaleFactor: 2.0}
pipeline = sciline.Pipeline(providers, params=params)
pipeline.visualize(Result, graph_attr={'rankdir': 'LR'})

`Filename` has no value.
A [Stage](../generated/classes/sciline.Stage.rst) cuts the pipeline at a set of *inputs*, keys whose values are supplied on each call, and computes the values of a set of *outputs* from them.
Everything the outputs need that does not depend on the inputs is computed once, on first use.
The stage holds the values of this part that the per-call part reads, its *frontier*:

In [ ]:
stage = sciline.Stage(pipeline, outputs=(Result,), inputs=(Filename,))
{
    f: stage.compute({Filename: f})[Result]
    for f in ['file102.txt', 'file103.txt', 'file104.txt']
}

`compute` was called three times, but the calibration was loaded once:

In [ ]:
calibration_loads

The stage tells us where it cut the pipeline.
`visualize` draws the part of the pipeline the stage uses, with the inputs, the part computed once, the held values, and the per-call part marked:

In [ ]:
stage.visualize(graph_attr={'rankdir': 'TD'})

`frontier` and `dynamic` are tuples of keys and `keys` is a frozenset.
`frontier` lists the held keys, and `static()` returns their values.
`dynamic` lists the keys whose values depend on the inputs and are computed on each call.
`keys` contains every key the stage uses, computed once or per call:

In [ ]:
stage.frontier

In [ ]:
stage.dynamic

In [ ]:
stage.keys

An input does not have to be a parameter.
If it is an intermediate result, its provider and everything upstream of it are cut off, and the stage computes only what lies downstream.
Here `Total` is supplied, so nothing is loaded and only `ScaleFactor` is held; the cut-off keys are not in `keys` either:

In [ ]:
from_total = sciline.Stage(pipeline, outputs=(Result,), inputs=(Total,))
from_total.frontier, from_total.dynamic, from_total.keys

In [ ]:
from_total.compute({Total: 10.0})

A stage is a snapshot of the pipeline at the time it is built.
Set all parameters, except the inputs, before building it; changing the pipeline afterwards does not affect the stage.

## Building several stages together

Two stages built from the same pipeline may share work that each computes once.
Each stage computes this part on first use, so two stages used independently both load the calibration.
[build_stages](../generated/functions/sciline.build_stages.rst) builds several stages from one pipeline, one for each tuple of outputs, and computes this part of all of them in one run, so shared work is done once.
`inputs` is shared: every stage is cut at each of these keys and takes those that its outputs need.
It returns the stages in the order of the outputs:

In [ ]:
calibration_loads.clear()
calibrated, totals = sciline.build_stages(
    pipeline, inputs=(Filename,), outputs=[(Calibrated,), (Total,)]
)
calibration_loads

Each stage holds only the values at its own frontier, available from `static()`:

In [ ]:
calibrated.static(), totals.static()

## Combining over files

To combine over files, we need two stages: one from `Filename` to the key at which we combine, the *accumulation key*, and one from that key to the result.
Between them, an accumulator combines the values computed per file.
An accumulator is any object with a `push` method and a `value` property, see [Accumulator](../generated/classes/sciline.Accumulator.rst).

Here we combine `Calibrated` by concatenation.
[Buffered](../generated/classes/sciline.Buffered.rst) makes accumulators that keep every pushed value and apply a function to all of them when the combined value is read:

In [ ]:
def concat(*lists: list) -> list:
    return [x for values in lists for x in values]


def combine_files(contribute, finalize, accumulators, filenames):
    acc = {key: make() for key, make in accumulators.items()}
    for filename in filenames:
        values = contribute.compute({Filename: filename})
        for key in contribute.dynamic_outputs:
            acc[key].push(values[key])
    return finalize.compute({key: a.value for key, a in acc.items()})


files = ['file102.txt', 'file103.txt', 'file104.txt', 'file105.txt']
contribute = sciline.Stage(pipeline, outputs=(Calibrated,), inputs=(Filename,))
finalize = sciline.Stage(pipeline, outputs=(Result,), inputs=(Calibrated,))
combine_files(contribute, finalize, {Calibrated: sciline.Buffered(concat)}, files)

`Calibrated` was computed per file, the four lists were concatenated, and `Total` and `Result` were computed once from the concatenated list.
`combine_files` is the *driver*: the code that calls the stages, here in a loop over files, and pushes their values into accumulators.
It pushes `dynamic_outputs`, the outputs that depend on the inputs.
The section "Push only what depends on the files" below explains why.
`Buffered(concat)` is a factory: calling it gives a new, empty accumulator.
`combine_files` calls the factories each time it runs, so a second call does not add to the values of the first.

`Buffered` keeps every pushed value.
For a sum of large arrays over many files, a running total is much cheaper.
[Reduced](../generated/classes/sciline.Reduced.rst) makes accumulators that apply a binary function to the result so far and each pushed value, and keep only the result.
`Reduced(operator.add)` is a factory too.
The function must be associative: a combined value can be pushed again, so values may be combined in groups, see "The steps separately".
It must not modify its arguments.
Here we combine at `Total` instead, and sum as we go:

In [ ]:
contribute = sciline.Stage(pipeline, outputs=(Total,), inputs=(Filename,))
finalize = sciline.Stage(pipeline, outputs=(Result,), inputs=(Total,))
combine_files(contribute, finalize, {Total: sciline.Reduced(operator.add)}, files)

The result is the same, but only one number per file is pushed, and the accumulator keeps only the running total.

### Push only what depends on the files

`combine_files` pushes `dynamic_outputs`, not all outputs of the stage.
An output that does not depend on the inputs, such as `Calibration`, is computed once and returned by every call.
Pushed into a sum, it would be counted once per file:

In [ ]:
with_calibration = sciline.Stage(
    pipeline, outputs=(Total, Calibration), inputs=(Filename,)
)
with_calibration.outputs, with_calibration.dynamic_outputs

The final stage takes only the dynamic outputs as inputs.
It holds `Calibration`, computed once:

In [ ]:
finalize_with_calibration = sciline.Stage(
    pipeline,
    outputs=(Result, Calibration),
    inputs=with_calibration.dynamic_outputs,
)
combine_files(
    with_calibration,
    finalize_with_calibration,
    {Total: sciline.Reduced(operator.add)},
    files,
)

## The steps separately

The loop in `combine_files` has three steps: compute a contribution per file, push its values into accumulators, and compute the result from the combined values.
They can also be called separately.
A contribution is a plain dictionary, so the steps can run in separate processes, with the contributions serialized between them:

In [ ]:
def combine(contributions):
    acc = sciline.Reduced(operator.add)()
    for contribution in contributions:
        acc.push(contribution[Total])
    return {Total: acc.value}


contributions = {f: contribute.compute({Filename: f}) for f in files}
finalize.compute(combine(contributions.values()))

Combined values can be pushed again, so combining can also proceed in groups.
The driver can keep the contributions in a dict keyed by file, to add or remove files without recomputing the others.
Here we drop one file:

In [ ]:
del contributions['file105.txt']
finalize.compute(combine(contributions.values()))

## Files in parallel

`contribute.compute` is a plain function of the values of the inputs, so any executor can run it.
Sciline does not run anything in parallel itself; the driver decides how to iterate over the files.
A stage can be called from several threads at once, and the part computed once is still computed only once:

In [ ]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor() as executor:
    contributions = executor.map(lambda f: contribute.compute({Filename: f}), files)
    combined = combine(contributions)
finalize.compute(combined)

## Results per file without combining

To compute the value of a key for each file without combining, call a stage in a loop.
The `calibrated` stage from above holds the calibration and computes `Calibrated` per file:

In [ ]:
{f: calibrated.compute({Filename: f})[Calibrated] for f in files}